# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
%%capture
!pip install datasets evaluate transformers


In [ ]:
# Import PyTorch library for tensor operations and neural network computations
import torch
# Import AdamW optimizer - an improved version of Adam optimizer with weight decay regularization
from torch.optim import AdamW
# Import Hugging Face transformers: AutoTokenizer for text preprocessing and AutoModelForSequenceClassification for classification tasks
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Same as before
# Define the pre-trained model checkpoint name (BERT base model, uncased version)
checkpoint = "bert-base-uncased"
# Load the tokenizer associated with the checkpoint - converts text to token IDs and handles special tokens
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# Load the pre-trained BERT model configured for sequence classification tasks (binary/multi-class classification)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
# Define a list of two sample sentences to be processed
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
# Tokenize the sequences: padding=True ensures all sequences have the same length (pads shorter sequences),
# truncation=True cuts sequences longer than the model's max length, return_tensors="pt" returns PyTorch tensors
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
# Add ground truth labels to the batch - labels are needed for training (1, 1 means both are positive class)
# In a real scenario, these would be actual labels from your training data
batch["labels"] = torch.tensor([1, 1])

# Initialize the AdamW optimizer with the model's parameters - this will update model weights during training
optimizer = AdamW(model.parameters())
# Forward pass: feed the batch (tokenized inputs + labels) to the model and compute the loss
# The **batch unpacks the dictionary (input_ids, attention_mask, labels, etc.) as keyword arguments
loss = model(**batch).loss
# Backward pass: compute gradients of the loss with respect to all model parameters using automatic differentiation
loss.backward()
# Update step: adjust model weights using the computed gradients according to the optimizer's algorithm
optimizer.step()

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

In [ ]:
raw_train_dataset.features

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_sentences_1 = tokenizer(raw_datasets["train"]["sentence1"])
tokenized_sentences_2 = tokenizer(raw_datasets["train"]["sentence2"])

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

In [ ]:
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"],
    raw_datasets["train"]["sentence2"],
    padding=True,
    truncation=True,
)

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}